In [1]:
import sys

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
sys.path.append("..")

In [4]:
from module1.ingest import load_faq_data

In [5]:
documents = load_faq_data()

In [6]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [7]:
documents = documents_llm

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [10]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
response.output_parsed.questions

['I found this course late — can I still join now, or is it too late?',
 'If I start the course after it began, am I still allowed to take part?',
 'Is late enrollment okay, and what do I need to do if I want the certificate?',
 'Can I join the course midway and still get credited for it?',
 'What’s the deadline for joining if I want to be eligible for the certificate?']

In [16]:
from evaluation_utils import llm_structured, llm_structured_retry

In [17]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I found this course late — can I still start and join in now?', 'Am I allowed to enroll after the course has already started?', 'If I join the course late, can I still get a certificate?', 'What do I need to do to qualify for the certificate if I’m joining now?', 'Is the project deadline the only thing I need to worry about for getting certified if I’m new to the course?']


In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I found this course late — can I still start and join in now?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to enroll after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course late, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for the certificate if I’m joining now?',
  'document': '74eb249bbf'},
 {'question': 'Is the project deadline the only thing I need to worry about for getting certified if I’m new to the course?',
  'document': '74eb249bbf'}]

In [19]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
from tqdm import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:07<00:00,  1.52s/it]


In [22]:
ground_truth[:5]

[{'question': 'Can I still join the course if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start this course, or can I still enroll?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I missed the start date—can I still take the course and earn a certificate?',
  'document': '74eb249bbf'}]

In [23]:
len(ground_truth)

25

In [29]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]